Data Integration: Merge the two different versions of the ISIC dataset from 2017 and 2018 into a unified folder structure and standardize the filename format.

In [ ]:
import os
import shutil

# ================= Configure source paths =================
# Please verify your specific folder names
DIR_2017_IMG = "/gpfs/work/bio/yixuanli2204/FYP_Project/data/vmunt_train_base/images_raw/17"
DIR_2017_MSK = "/gpfs/work/bio/yixuanli2204/FYP_Project/data/vmunt_train_base/masks_raw/masks_lesion_2017"

DIR_2018_IMG = "/gpfs/work/bio/yixuanli2204/FYP_Project/data/vmunt_train_base/images_raw/17"
DIR_2018_MSK = "/gpfs/work/bio/yixuanli2204/FYP_Project/data/vmunt_train_base/masks_raw/masks_lesion_2018"

# ================= Configure destination paths =================
DEST_ROOT = "/gpfs/work/bio/yixuanli2204/FYP_Project/data/vmunt_train_base"
DEST_IMG = os.path.join(DEST_ROOT, "images")
DEST_MSK = os.path.join(DEST_ROOT, "masks_lesion")

os.makedirs(DEST_IMG, exist_ok=True)
os.makedirs(DEST_MSK, exist_ok=True)

def process_year(year, img_dir, msk_dir):
    print(f"Processing {year} data...")
    count = 0
    
    # Iterate through source image directory
    for fname in os.listdir(img_dir):
        # 【Key correction】: Only process .jpg files
        if not fname.lower().endswith(".jpg"):
            continue
            
        # 1. Process image filename
        # ISIC_0000000.jpg -> ISIC2017_0000000.jpg
        new_name = fname.replace("ISIC_", f"ISIC{year}_")
        
        # 2. Process corresponding mask filename
        # 2017 format: ISIC_0000000_segmentation.png
        # 2018 format: ISIC_0000000_segmentation.png (typically same)
        old_id = os.path.splitext(fname)[0]
        msk_name = f"{old_id}_segmentation.png"
        
        # Target mask name unified as: ISIC2017_0000000_lesion.png
        new_msk_name = new_name.replace(".jpg", "_lesion.png")
        
        src_img_path = os.path.join(img_dir, fname)
        src_msk_path = os.path.join(msk_dir, msk_name)
        
        # 3. Only copy if mask also exists
        if os.path.exists(src_msk_path):
            shutil.copy(src_img_path, os.path.join(DEST_IMG, new_name))
            shutil.copy(src_msk_path, os.path.join(DEST_MSK, new_msk_name))
            count += 1
            
    print(f"{year} processing completed, merged {count} sample pairs.")

# Execute merging
process_year("2017", DIR_2017_IMG, DIR_2017_MSK)
process_year("2018", DIR_2018_IMG, DIR_2018_MSK)

print(f"\nSummary completed! Total samples: {len(os.listdir(DEST_IMG))}")